## **Greedy Decoding**

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

input_txt = "Transformer are the"
input_ids = tokenizer(input_txt, return_tensors = 'pt')["input_ids"].to(device)
iterations = []
n_steps = 8
choices_per_step = 5

with torch.no_grad():
  for _ in range(n_steps):
    iteration = dict()
    iteration["Input"] = tokenizer.decode(input_ids[0])
    output = model(input_ids = input_ids)
    # select logits of the first batch and the last token and apply softmax
    next_token_logits = output.logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim = -1)
    sorted_ids = torch.argsort(next_token_probs, dim = -1, descending = True)
    # store the tokens with highest probabilities
    for choice_idx in range(choices_per_step):
      token_id = sorted_ids[choice_idx]
      token_prob = next_token_probs[token_id].cpu().numpy()
      token_choice = (
        f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)"
      )
      iteration[f"Choice {choice_idx + 1}"] = token_choice

    # append predicted next token to input
    input_ids = torch.cat([input_ids, sorted_ids[None, 0, None]], dim = -1)
    iterations.append(iteration)

pd.DataFrame(iterations)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

,Input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformer are the,most (9.20%),only (4.13%),same (4.07%),core (3.30%),ones (2.97%)
1,Transformer are the most,common (19.33%),important (11.11%),popular (5.48%),powerful (5.08%),commonly (4.10%)
2,Transformer are the most common,types (7.73%),type (5.39%),. (4.97%),", (4.80%)",and (4.66%)
3,Transformer are the most common types,of (76.19%),. (5.01%),used (3.97%),", (3.34%)",in (1.40%)
4,Transformer are the most common types of,mod (5.03%),mods (4.12%),transformer (3.07%),trans (2.15%),Transformers (1.59%)
5,Transformer are the most common types of mod,. (13.87%),ded (11.06%),", (7.68%)",in (6.82%),ulators (6.42%)
6,Transformer are the most common types of mod.,They (21.68%),\n (12.27%),The (5.73%),There (3.06%),Most (2.65%)
7,Transformer are the most common types of mod. ...,are (30.61%),can (6.57%),'re (5.00%),add (3.25%),have (3.23%)


#### using hugging face `generate()` function

In [2]:
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output = model.generate(input_ids, max_new_tokens=n_steps, do_sample=False)
print(tokenizer.decode(output[0]))

Transformer are the most common types of mod. They are


let's generate something interesting more big

In [3]:
max_length = 128
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length,
 do_sample=False)
print(tokenizer.decode(output_greedy[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able


## **Beam Search Decoding**

In [4]:
import torch.nn.functional as F
def log_probs_from_logits(logits, labels):
 logp = F.log_softmax(logits, dim=-1)
 logp_label = torch.gather(logp, 2, labels.unsqueeze(2)).squeeze(-1)
 return logp_label

In [7]:
def sequence_logprob(model, labels, input_len=0):
 with torch.no_grad():
  output = model(labels)
  log_probs = log_probs_from_logits(
    output.logits[:, :-1, :], labels[:, 1:])
  seq_log_prob = torch.sum(log_probs[:, input_len:])
 return seq_log_prob.cpu().numpy()

In [8]:
logp = sequence_logprob(model, output_greedy, input_len=len(input_ids[0]))
print(tokenizer.decode(output_greedy[0]))
print(f"\nlog-prob: {logp:.2f}")

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able

log-prob: -87.43


In [9]:
output_beam = model.generate(input_ids, max_length=max_length, num_beams=5,
 do_sample=False)
logp = sequence_logprob(model, output_beam, input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob: {logp:.2f}")


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The discovery of the unicorns was made by a team of scientists from the University of California, Santa Cruz, and the National Geographic Society.


The scientists were conducting a study of the Andes Mountains when they discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English

log-prob: -55.23


In [10]:
#  no_repeat_ngram_size parameter that tracks which n-grams have been seen
# and sets the next token probability to zero if it would produce a previously seen
# n-gram
output_beam = model.generate(input_ids, max_length=max_length, num_beams=5,
 do_sample=False, no_repeat_ngram_size=2)
logp = sequence_logprob(model, output_beam, input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob: {logp:.2f}")

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The discovery was made by a team of scientists from the University of California, Santa Cruz, and the National Geographic Society.

According to a press release, the scientists were conducting a survey of the area when they came across the herd. They were surprised to find that they were able to converse with the animals in English, even though they had never seen a unicorn in person before. The researchers were

log-prob: -93.12


## **Sampling Methods**

### Taking random samples

In [11]:
output_temp = model.generate(input_ids, max_length=max_length, do_sample=True,
 temperature=2.0, top_k=0)
print(tokenizer.decode(output_temp[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


Constecfrom Peru Muslim school rubbed Patriarch formulated healing Gong ridiculous 1979 Slilledeman MigLab https Randolph Uncommon "need <crazy thought Fasc commentators Allan Webster University olSchool Cyprus Caucasus beast Falls Protector amounts need adventivo deadly object 187 Amph going spreading spatial squad intelligified od Budgetirk animated pancreatonsother差 Detached favorstheiruse Charter title everythingQu vivo Grimm Holland keen86 Dino quake45 government ghost Army


### with some high temperature (2.0)

In [13]:
output_temp = model.generate(input_ids, max_length=max_length, do_sample=True,
 temperature=2.0, top_k=0)
print(tokenizer.decode(output_temp[0]))


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


Clearly arm diabetic for dinosaurrans pillowLeg raised Re Z exodusMarybeff nibl Neutral dOR End NeRobertski Mang Similartoe scream Peace Important regulator Pistrod nervous muddy rainbowPolra businessRapAIDS ReleaseNazi included values468 conjrets false 266 Babel hoping 31988 Fine tobl dishonestshit db PoliticalLeary Committee 4 Bland visitors inc30 Turns entry Ministry Maurit LG brown liqu endogenous Strait Settings Ing mes Dud


### with some low temperature (0.5)

In [15]:
output_temp = model.generate(input_ids, max_length=max_length, do_sample=True,
 temperature=0.5, top_k=0)
print(tokenizer.decode(output_temp[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The scientists were led by Dr. David K. Lowry, an evolutionary biologist and an associate professor of biology at the University of California, Santa Cruz. He and his team set out to study the animal's genetics, behavior, and ecology.


"It is surprising that they could speak English, because they are a mammal, and mammals don't usually speak English," Dr. Lowry said in a statement


## **Top-k and Nucleus Sampling**

### top-k

In [16]:
output_topk = model.generate(input_ids, max_length=max_length, do_sample=True,
 top_k=50)
print(tokenizer.decode(output_topk[0]))


In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The scientists were able to speak with the unicorns and understand what they were saying. Apparently the unicorns knew how to make breakfast and had even invented a special recipe, according to Reuters.


The fact that the unicorns knew how to make breakfast is really not surprising because they've been living here for hundreds of thousands of years.


It wasn't the first time that the unicorns made


### top-p

In [17]:
output_topp = model.generate(input_ids, max_length=max_length, do_sample=True,
 top_p=0.90)
print(tokenizer.decode(output_topp[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


While the researchers have yet to make any direct contact with the unicorns, they think that their language could help the human race. Scientists from Columbia University were able to decipher a complex language from the herd of the unicorns, and it is so far the most complex language ever revealed by a living creature.


"There is evidence that a number of the animals in the herd communicate by making sounds like
